# Exerc?cio: Classifica??o Bin?ria
## Objetivos: Exercitar os conceitos referente ? classifica??o bin?ria.

In [ ]:
import duckdb
import unicodedata
import re
from datasketch import MinHash, MinHashLSH
from tqdm import tqdm
from joblib import Parallel, delayed
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec
from scipy.sparse import hstack
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from tqdm import tqdm

### 1. Crie um arquivo Jupyter Notebook e realize as seguintes opera??es:

a. Ler o dataset fakeTelegram.BR_2022.csv

b. Remova os trava-zaps, as linhas repetidas (duplicadas) e textos com menos de 5 palavras.

c. Agrupe as linhas com postagens iguais ou extremamente semelhantes. Aqui voc? pode utilizar uma m?trica de semelhan?a de textos. Crie uma vari?vel para representar a quantidade de vezes que a mensagem foi compartilhada. Observe que ao agrupar linhas que possuem a ?mesma? postagem (texto), voc? deve escolher como valor para as vari?veis data e hora da postagem, os valores da c?pia mais antiga.

d. Voc? pode criar novos atributos num?ricos, tais como: quantidade de palavras, quantidade de caracteres etc.

In [ ]:
conn = duckdb.connect()

telegram_data = conn.read_csv("../data/fakeTelegram.BR_2022.csv")

query = """
    SELECT * FROM telegram_data
"""

df = conn.execute(query).fetchdf()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df = conn.execute(f"""
    SELECT * 
    FROM telegram_data
    WHERE trava_zap IS NOT TRUE
""").fetch_df()

In [ ]:
df.shape

In [ ]:
df_ = conn.execute("SELECT DISTINCT * FROM df").fetch_df()

In [ ]:
query = """
SELECT *
FROM df_
WHERE array_length(string_split(text_content_anonymous, ' ')) >= 5
"""

df = conn.execute(query).fetch_df()

df.shape

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Converte para min?sculas
    text = text.lower()
    
    # 2. Remover URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Remover caracteres ?rabes
    text = re.sub(r'[\u0600-\u06FF]+', '', text)
    
    # 4. Remover acentos (Normaliza??o)
    text = unicodedata.normalize('NFKD', text)\
           .encode('ascii', 'ignore')\
           .decode('utf-8')
    
    # 5. Remover pontua??o e caracteres especiais
    text = re.sub(r'[^\w\s]', '', text)
    
    # 6. Remover emojis 
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # s?mbolos & pictogramas
        u"\U0001F680-\U0001F6FF"  # transporte & s?mbolos
        u"\U0001F1E0-\U0001F1FF"  # bandeiras (iOS)
                           "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # 7. Remover espa?os extras (no in?cio, fim e m?ltiplos espa?os no meio)
    text = ' '.join(text.split())
    
    return text

df['cleaned_text'] = df['text_content_anonymous'].apply(clean_text)

In [ ]:
def get_shingles(text, k=5):
    """Cria um conjunto de shingles de tamanho k a partir de um texto."""
    if not isinstance(text, str) or len(text) < k:
        return set()
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def create_minhash(text, num_perm, shingle_size):
    mh = MinHash(num_perm=num_perm)
    shingles = get_shingles(text, k=shingle_size)
    if not shingles:
        return None
    for shingle in shingles:
        mh.update(shingle.encode('utf-8'))
    return mh

In [ ]:
print("Iniciando agrupamento de textos semelhantes...")

JACCARD_THRESHOLD = 0.7
NUM_PERMUTATIONS = 128
SHINGLE_SIZE = 5

texts = df['cleaned_text'].tolist()

print("Etapa 1 de 4: Criando MinHashes em paralelo...")
minhash_list = Parallel(n_jobs=-1)(
    delayed(create_minhash)(text, NUM_PERMUTATIONS, SHINGLE_SIZE) 
    for text in tqdm(texts, desc="Criando assinaturas")
)

# Remapeia a lista para o dicion?rio, o que ? r?pido
minhashes = {idx: mh for idx, mh in enumerate(minhash_list) if mh is not None}

print("\nEtapa 2 de 4: Indexando MinHashes no LSH...")
lsh = MinHashLSH(threshold=JACCARD_THRESHOLD, num_perm=NUM_PERMUTATIONS)
for idx, mh in tqdm(minhashes.items(), desc="Indexando no LSH"):
    lsh.insert(idx, mh)

print("\nEtapa 3 de 4: Consultando LSH e agrupando...")
groups = []
used = set()
for idx in tqdm(minhashes, desc="Consultando e agrupando"):
    if idx not in used:
        similar_indices = lsh.query(minhashes[idx])
        groups.append(similar_indices)
        used.update(similar_indices)

print(f"\nAgrupamento conclu?do. Foram encontrados {len(groups)} grupos de mensagens.")


print("\nEtapa 4 de 4: Consolidando resultados...")
grouped_data = []
for group in tqdm(groups, desc="Consolidando grupos"):
    group_df = df.iloc[list(group)]
    oldest = group_df.loc[group_df['date_message'].idxmin()]
    count = len(group_df)
    new_row = oldest.to_dict()
    new_row['shared_count'] = count
    grouped_data.append(new_row)

result_df = pd.DataFrame(grouped_data)
result_df = result_df.sort_values(by='shared_count', ascending=False)

print("\nProcesso finalizado com sucesso!")

In [ ]:
def extract_features(text):
    if not isinstance(text, str):
        return {
            'word_count': 0,
            'char_count': 0,
            'unique_word_ratio': 0,
            'avg_word_length': 0
        }
    
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    
    if word_count > 0:
        unique_words = set(words)
        unique_word_ratio = len(unique_words) / word_count
        avg_word_length = sum(len(word) for word in words) / word_count
    else:
        unique_word_ratio = 0
        avg_word_length = 0
    
    return {
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_ratio': unique_word_ratio,
        'avg_word_length': avg_word_length
    }

features = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))

result_df = pd.concat([result_df, features], axis=1)

print("Caracter?sticas num?ricas adicionadas com sucesso ao result_df.")
print(result_df[['cleaned_text', 'shared_count', 'word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']].head())

Utilizando os dados referente a postagens no Telegram, crie um modelo preditivo (classificador bin?rio) para classificar uma mensagem em duas classes poss?veis: ?viral? (classe positiva) ou ?n?o viral? (classe negativa). Para rotular as mensagens ?nicas (agrupadas) nas classes, ?viral? e ?n?o viral?, utilize a seguinte estrat?gia: Calcule um limiar (threshold). Por exemplo, mediana do n?mero de compartilhamentos mais dois desvios padr?es. As mensagens com quantidade de compartilhamentos maiores ou iguais ao limiar definido devem ser rotuladas como ?virais?. As demais mensagens devem ser rotuladas como ?n?o virais?.

A avalia??o experimental dever? considerar:

e. O algoritmo de classifica??o: regress?o log?stica, ?rvore de decis?o e uma estrat?gia baseada em ?ensemble?;

f. Regulariza??o: Com regulariza??o (Ridge, Lasso ou ElasticNet) e sem regulariza??o;

g. Normaliza??o dos dados: sem normaliza??o, Z-Score, Min-Max (OPCIONAL); 

h. Pr?-processamento de dados: sem pr?-processamento e com pr?-processamento;

i. Embedding: BOW, TF-IDF, Word2Vec;

j. N-Gramas: unigramas, bigramas, trigramas;

k. Treinamento, Valida??o e Teste: Outer K-Fold Cross-Validation;

In [ ]:
median_shares = result_df['shared_count'].median()
std_dev_shares = result_df['shared_count'].std()
viral_threshold = median_shares + (2 * std_dev_shares)

print(f"An?lise para definir 'viral':")
print(f"  - Mediana de compartilhamentos: {median_shares:.2f}")
print(f"  - Desvio Padr?o dos compartilhamentos: {std_dev_shares:.2f}")
print(f"  - Limiar de Viraliza??o (Mediana + 2*DP): {viral_threshold:.2f}")
print("-" * 30)

result_df['is_viral'] = np.where(result_df['shared_count'] >= viral_threshold, 1, 0)

class_distribution = result_df['is_viral'].value_counts(normalize=True) * 100

print("Distribui??o das Classes:")
print(f"  - N?o Viral (0): {class_distribution.get(0, 0):.2f}%")
print(f"  - Viral (1):     {class_distribution.get(1, 0):.2f}%")


print("\nExemplo de dados rotulados:")
print(result_df[['shared_count', 'is_viral']].sort_values(by='shared_count', ascending=False).head())

In [ ]:
!pip install imbalanced-learn

# e. O algoritmo de classifica??o: regress?o log?stica, ?rvore de decis?o e uma estrat?gia baseada em ?ensemble?;
# f. Regulariza??o: Com regulariza??o (Ridge, Lasso ou ElasticNet) e sem regulariza??o;

In [ ]:
features = [
    'word_count', 
    'char_count', 
    'avg_word_length',
    'unique_word_ratio',
]
X = result_df[features]
y = result_df['is_viral']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Distribui??o original no treino: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Distribui??o ap?s SMOTE no treino: {y_train_resampled.value_counts().to_dict()}")

models = {
    "Reg. Log. - Sem Regulariza??o": LogisticRegression(penalty=None, solver='saga', random_state=42, max_iter=1000),
    "Reg. Log. - L2 (Ridge)": LogisticRegression(penalty='l2', C=1.0, solver='saga', random_state=42, max_iter=1000),
    "Reg. Log. - L1 (Lasso)": LogisticRegression(penalty='l1', C=1.0, solver='saga', random_state=42, max_iter=1000),
    "?rvore de Decis?o": DecisionTreeClassifier(random_state=42),
    "Random Forest (Ensemble)": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    print(f"\n--- Treinando e Avaliando: {name} ---")
    
    model.fit(X_train_resampled, y_train_resampled)
    
    y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred, target_names=["N?o Viral (0)", "Viral (1)"]))

    if 'Reg. Log.' in name:
        print("   Coeficientes (import?ncia das features):")
        coefs = pd.Series(model.coef_[0], index=features)
        print(coefs.sort_values(ascending=False))

# item g ? opcional, pulei nessa lista
# h. Pr?-processamento de dados: sem pr?-processamento e com pr?-processamento;

In [ ]:
print("="*20)
print("CEN?RIO 1: SEM PR?-PROCESSAMENTO NUM?RICO")
print("="*20)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

models_sem_preproc = {
    "Regress?o Log?stica": LogisticRegression(random_state=42, max_iter=1000),
    "?rvore de Decis?o": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models_sem_preproc.items():
    print(f"\n--- Avaliando: {name} ---")
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["N?o Viral", "Viral"]))


print("\n" + "="*20)
print("CEN?RIO 2: COM PR?-PROCESSAMENTO NUM?RICO")
print("="*20)

def remove_outliers_iqr(X, y):
    X_clean = X.copy()
    y_clean = y.copy()
    for col in X_clean.columns:
        Q1 = X_clean[col].quantile(0.3)
        Q3 = X_clean[col].quantile(0.7)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        mask = (X_clean[col] >= lower_bound) & (X_clean[col] <= upper_bound)
        X_clean = X_clean.loc[mask]
        y_clean = y_clean.loc[mask]
    return X_clean, y_clean

X_train_no_outliers, y_train_no_outliers = remove_outliers_iqr(X_train, y_train)
print(f"Removendo outliers do treino: {len(X_train)} -> {len(X_train_no_outliers)} amostras")

X_train_proc_smote, y_train_proc_smote = smote.fit_resample(X_train_no_outliers, y_train_no_outliers)

pipelines_com_preproc = {
    "Regress?o Log?stica": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    "?rvore de Decis?o": Pipeline([ 
        ('classifier', DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ('classifier', RandomForestClassifier(random_state=42))
    ])
}

for name, pipeline in pipelines_com_preproc.items():
    print(f"\n--- Avaliando: {name} ---")
    pipeline.fit(X_train_proc_smote, y_train_proc_smote)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["N?o Viral", "Viral"]))


# i. Embedding: BOW, TF-IDF, Word2Vec;

In [ ]:
features_df = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))
result_df = pd.concat([result_df, features_df], axis=1)


text_feature = 'cleaned_text'
numeric_features = ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']

X = result_df[numeric_features + [text_feature]]
y = result_df['is_viral']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

models_to_run = {
    "Regress?o Log?stica": LogisticRegression(penalty='l2', C=1.0, solver='saga', random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1)
}

binary_label_names = ["N?o Viral", "Viral"]

print("\n" + "="*60)
print("INICIANDO CEN?RIO 1: TF-IDF (CLASSIFICA??O BIN?RIA)")
print("="*60)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_text_tfidf = tfidf_vectorizer.fit_transform(X_train[text_feature])
X_test_text_tfidf = tfidf_vectorizer.transform(X_test[text_feature])

scaler_tfidf = StandardScaler()
X_train_num_scaled_tfidf = scaler_tfidf.fit_transform(X_train[numeric_features])
X_test_num_scaled_tfidf = scaler_tfidf.transform(X_test[numeric_features])

X_train_final_tfidf = hstack([X_train_num_scaled_tfidf, X_train_text_tfidf])
X_test_final_tfidf = hstack([X_test_num_scaled_tfidf, X_test_text_tfidf])

print("Aplicando SMOTE para TF-IDF...")
smote_tfidf = SMOTE(random_state=42)
X_train_resampled_tfidf, y_train_resampled_tfidf = smote_tfidf.fit_resample(X_train_final_tfidf, y_train)
print("SMOTE conclu?do.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com TF-IDF"):
    model.fit(X_train_resampled_tfidf, y_train_resampled_tfidf)
    y_pred = model.predict(X_test_final_tfidf)
    print(f"\n--- Resultado para: {name} com TF-IDF ---")
    print(classification_report(y_test, y_pred, target_names=binary_label_names, zero_division=0))


print("\n" + "="*60)
print("INICIANDO CEN?RIO 2: BAG-OF-WORDS (BOW) (CLASSIFICA??O BIN?RIA)")
print("="*60)

bow_vectorizer = CountVectorizer(max_features=5000)
X_train_text_bow = bow_vectorizer.fit_transform(X_train[text_feature])
X_test_text_bow = bow_vectorizer.transform(X_test[text_feature])

scaler_bow = StandardScaler()
X_train_num_scaled_bow = scaler_bow.fit_transform(X_train[numeric_features])
X_test_num_scaled_bow = scaler_bow.transform(X_test[numeric_features])

X_train_final_bow = hstack([X_train_num_scaled_bow, X_train_text_bow])
X_test_final_bow = hstack([X_test_num_scaled_bow, X_test_text_bow])

print("Aplicando SMOTE para BOW...")
smote_bow = SMOTE(random_state=42)
X_train_resampled_bow, y_train_resampled_bow = smote_bow.fit_resample(X_train_final_bow, y_train)
print("SMOTE conclu?do.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com BOW"):
    model.fit(X_train_resampled_bow, y_train_resampled_bow)
    y_pred = model.predict(X_test_final_bow)
    print(f"\n--- Resultado para: {name} com BOW ---")
    print(classification_report(y_test, y_pred, target_names=binary_label_names, zero_division=0))


print("\n" + "="*60)
print("INICIANDO CEN?RIO 3: WORD2VEC (CLASSIFICA??O BIN?RIA)")
print("="*60)

tokenized_text_train = [text.split() for text in X_train[text_feature]]
vector_size = 100
print("Treinando modelo Word2Vec...")
w2v_model = Word2Vec(sentences=tokenized_text_train, vector_size=vector_size, window=5, min_count=2, workers=4)
print("Modelo Word2Vec treinado.")

def document_vector(doc, model, num_features):
    doc_vector = np.zeros((num_features,), dtype="float32")
    num_words = 0
    words = doc.split()
    for word in words:
        if word in model.wv:
            num_words += 1
            doc_vector = np.add(doc_vector, model.wv[word])
    if num_words > 0:
        doc_vector = np.divide(doc_vector, num_words)
    return doc_vector

X_train_text_w2v = np.array([document_vector(doc, w2v_model, vector_size) for doc in tqdm(X_train[text_feature], desc="Criando vetores de treino (W2V)")])
X_test_text_w2v = np.array([document_vector(doc, w2v_model, vector_size) for doc in tqdm(X_test[text_feature], desc="Criando vetores de teste (W2V)")])

scaler_w2v = StandardScaler()
X_train_num_scaled_w2v = scaler_w2v.fit_transform(X_train[numeric_features])
X_test_num_scaled_w2v = scaler_w2v.transform(X_test[numeric_features])

X_train_final_w2v = np.hstack([X_train_num_scaled_w2v, X_train_text_w2v])
X_test_final_w2v = np.hstack([X_test_num_scaled_w2v, X_test_text_w2v])

print("Aplicando SMOTE para Word2Vec...")
smote_w2v = SMOTE(random_state=42)
X_train_resampled_w2v, y_train_resampled_w2v = smote_w2v.fit_resample(X_train_final_w2v, y_train)
print("SMOTE conclu?do.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com Word2Vec"):
    model.fit(X_train_resampled_w2v, y_train_resampled_w2v)
    y_pred = model.predict(X_test_final_w2v)
    print(f"\n--- Resultado para: {name} com Word2Vec ---")
    print(classification_report(y_test, y_pred, target_names=binary_label_names, zero_division=0))

# j. N-Gramas: unigramas, bigramas, trigramas;

In [ ]:
X = result_df[numeric_features + [text_feature]] 
y = result_df['is_viral']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

ngram_configs = {
    "Apenas Unigramas": (1, 1),
    "Unigramas e Bigramas": (1, 2),
    "Unigramas, Bigramas e Trigramas": (1, 3)
}

for name, ngram_range in ngram_configs.items():
    print("="*40)
    print(f"TESTANDO CONFIGURA??O: {name}")
    print("="*40)
    
    print(f"Criando features TF-IDF com ngram_range={ngram_range}...")
    tfidf_vectorizer = TfidfVectorizer(
        ngram_range=ngram_range, 
        max_features=5000
    )
    X_train_text = tfidf_vectorizer.fit_transform(X_train[text_feature])
    X_test_text = tfidf_vectorizer.transform(X_test[text_feature])

    print("Combinando features num?ricas e de texto...")
    scaler = StandardScaler()
    X_train_num_scaled = scaler.fit_transform(X_train[numeric_features])
    X_test_num_scaled = scaler.transform(X_test[numeric_features])
    X_train_final = hstack([X_train_num_scaled, X_train_text])
    X_test_final = hstack([X_test_num_scaled, X_test_text])

    print("Rebalanceando o conjunto de treino com SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_final, y_train)

    print("Treinando o modelo Random Forest...")
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train_resampled, y_train_resampled)
    
    print(f"\n--- Resultado para: {name} ---")
    y_pred = model.predict(X_test_final)
    print(classification_report(y_test, y_pred, target_names=["N?o Viral", "Viral"]))

# k. Treinamento, Valida??o e Teste: Outer K-Fold Cross-Validation;

In [ ]:
numeric_features = ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']


cols_to_drop = [col for col in numeric_features if col in result_df.columns]
if cols_to_drop:
    print(f"Limpando colunas num?ricas pr?-existentes para evitar duplica??o: {cols_to_drop}")
    result_df = result_df.drop(columns=cols_to_drop)

features_df = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))
result_df = pd.concat([result_df, features_df], axis=1)
print("Features num?ricas criadas/atualizadas com sucesso.")


print("\nIniciando a Valida??o Cruzada Aninhada (Outer K-Fold)...")

text_feature = 'cleaned_text'
X = result_df[numeric_features + [text_feature]]
y = result_df['is_viral']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('text', TfidfVectorizer(ngram_range=(1, 2), max_features=5000), text_feature)
    ],
    remainder='drop'
)

full_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(solver='saga', random_state=42, max_iter=2000))
])

param_grid = {'classifier__C': [0.1, 1.0, 10]}

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

clf = GridSearchCV(
    estimator=full_pipeline, 
    param_grid=param_grid, 
    cv=inner_cv,
    scoring='f1_macro',
    n_jobs=-1
)

outer_scores = []
for i, (train_idx, test_idx) in enumerate(tqdm(outer_cv.split(X, y), total=outer_cv.get_n_splits(), desc="Outer CV Folds")):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    clf.fit(X_train, y_train)
    
    best_model = clf.best_estimator_
    y_pred = best_model.predict(X_test)
    score = f1_score(y_test, y_pred, pos_label=1)
    outer_scores.append(score)
    
    print(f"\nFold Externo {i+1}: Melhor C={clf.best_params_['classifier__C']}, F1-Score (Viral) = {score:.4f}")

print("\n" + "="*40)
print("Resultado Final da Valida??o Cruzada Aninhada")
print("="*40)
print(f"F1-Scores para a classe 'Viral' em cada um dos 5 folds: {np.round(outer_scores, 4)}")
print(f"M?dia do F1-Score: {np.mean(outer_scores):.4f}")
print(f"Desvio Padr?o do F1-Score: {np.std(outer_scores):.4f}")